# Ask My Pension: training on Colab

This notebook fine-tunes the retriever and the reader, and evaluates every model, on Colab's free GPU.
All the logic lives in the repository's `scripts/` folder; these cells only call it.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells from top to bottom.

What you need: `data/processed/passages.jsonl` from your own machine (built by `scripts/build_passages.py`).
It is not in git, because it holds the publishers' text.

At the end, download `results.zip` and copy its contents into the repository's `results/` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Set up

Clone the repository and install only what training and evaluation need.

In [ ]:
!git clone -q https://github.com/michael-alu/ask-my-pension.git
%cd ask-my-pension
!pip install -q "transformers>=4.44,<5" "sentence-transformers>=3,<5" "huggingface-hub<1.0" "datasets>=3,<4" accelerate rank-bm25

## 2. Upload the passages

Choose `data/processed/passages.jsonl` from your machine. It should hold 1,559 passages.

In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()

Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("data/processed/passages.jsonl").write_bytes(uploaded["passages.jsonl"])

passage_count = len(Path("data/processed/passages.jsonl").read_text().splitlines())
print(passage_count, "passages")
assert passage_count == 1559, "This is not the passages file the annotations were made on."

In [ ]:
!python scripts/export_squad.py

## 3. Retriever

Fine-tune both retrievers on the train split, then evaluate them next to their off-the-shelf versions.
Each evaluation adds rows to `results/experiments.md`.

In [ ]:
!python scripts/train_retriever.py BAAI/bge-small-en-v1.5 1.0 3
!python scripts/train_retriever.py sentence-transformers/all-MiniLM-L6-v2 1.0 3

In [ ]:
!python scripts/evaluate_dense.py BAAI/bge-small-en-v1.5
!python scripts/evaluate_dense.py models/retriever-bge-small-en-v1.5-100pct
!python scripts/evaluate_dense.py sentence-transformers/all-MiniLM-L6-v2
!python scripts/evaluate_dense.py models/retriever-all-MiniLM-L6-v2-100pct

## 4. Reader

Fine-tune both readers, then evaluate one change at a time:
- BM25 + zero-shot reader is the baseline (already in the table).
- Fine-tuned retriever + zero-shot reader: the effect of the retriever alone.
- BM25 + fine-tuned reader: the effect of the reader alone.
- Fine-tuned retriever + fine-tuned reader: the full system.

In [ ]:
!python scripts/train_reader.py deepset/roberta-base-squad2 1.0 2
!python scripts/train_reader.py deepset/tinyroberta-squad2 1.0 2

In [ ]:
!python scripts/evaluate_reader.py deepset/roberta-base-squad2 models/retriever-bge-small-en-v1.5-100pct
!python scripts/evaluate_reader.py models/reader-roberta-base-squad2-100pct bm25
!python scripts/evaluate_reader.py models/reader-roberta-base-squad2-100pct models/retriever-bge-small-en-v1.5-100pct
!python scripts/evaluate_reader.py deepset/tinyroberta-squad2 bm25
!python scripts/evaluate_reader.py models/reader-tinyroberta-squad2-100pct models/retriever-bge-small-en-v1.5-100pct

## 5. How much does more training data help?

Train on 25%, 50% and 75% of the train split (100% is above), then evaluate each.
If the scores are still climbing at 100%, more annotation would help.

In [ ]:
for share in ["0.25", "0.5", "0.75"]:
    !python scripts/train_retriever.py BAAI/bge-small-en-v1.5 {share} 3
    !python scripts/train_reader.py deepset/roberta-base-squad2 {share} 2

In [ ]:
for percent in ["25", "50", "75"]:
    !python scripts/evaluate_dense.py models/retriever-bge-small-en-v1.5-{percent}pct
    !python scripts/evaluate_reader.py models/reader-roberta-base-squad2-{percent}pct bm25

## 6. Download the results

Copy everything inside `results.zip` into the repository's `results/` folder, then commit it.

In [ ]:
!cat results/experiments.md
!zip -q -r results.zip results
files.download("results.zip")

## 7. (Optional, for the app) Keep the best models

The models disappear when this Colab session ends. The app needs the fine-tuned retriever and reader,
so download them now. This takes a few minutes, as the reader is about 500 MB.

In [ ]:
!zip -q -r best_models.zip models/retriever-bge-small-en-v1.5-100pct models/reader-roberta-base-squad2-100pct
files.download("best_models.zip")